# Chess AI — Self-Play Only (Kaggle, T4x2)

**Dùng khi:** Đã có model (từ PGN training hoặc self-play trước đó),
muốn tiếp tục cải thiện bằng self-play mà **không** cần train PGN lại.

## Cách chạy (train self-play từ đầu bằng model đã train PGN)

1. Trên Kaggle, add 2 dataset ở phần **Add Input**:
   - `chess-ai-source` — source code (engine C++, `colab_selfplay_pipeline.py`, ...)
   - `chess-model` — chứa `best_model_traced.pt` xuất ra từ notebook train PGN của bạn
2. Bật **GPU T4 x2** trong Settings → Accelerator.
3. Chạy tuần tự **Cell 1 → Cell 2 → Cell 3**.
   - Cell 2 sẽ tự tìm và in ra model PGN tìm được (`[Bootstrap] ✓ Model tìm thấy: ...`).
     Kiểm tra đúng file bạn muốn dùng trước khi qua Cell 3.
   - `SP_FORCE_FRESH_START = True` (mặc định) đảm bảo generation luôn bắt đầu
     lại từ 1 với model PGN làm điểm xuất phát, kể cả khi bạn Rerun Cell 1→3
     nhiều lần trong cùng session (output cũ tự động dời sang
     `_previous_run_<timestamp>/`, không bị mất/đè).
4. Theo dõi log mỗi generation:
   - `[SelfPlay] ...` — sinh dữ liệu tự chơi (chạy song song trên cả 2 GPU).
   - `[Arena] ... → PROMOTED ✓` hoặc `→ REJECTED ✗` — model vừa train có thắng
     model tốt nhất hiện tại không (xem mục Arena Gating bên dưới).

### Datasets cần add
| Dataset | Chứa gì | Dùng để |
|---------|---------|---------|
| `chess-ai-source` | Source code (engine C++, pipeline.py) | bắt buộc |
| `chess-model` | `best_model_traced.pt` từ lần train PGN/self-play trước | làm điểm khởi đầu (INITIAL_MODEL) |

> Nếu không add `chess-model`, pipeline sẽ tự tạo model ngẫu nhiên (chơi rất tệ lúc đầu).

## Arena Gating (mới)

Sau mỗi generation, model vừa train phải **đấu `SP_ARENA_GAMES` ván** với
model tốt nhất hiện tại (không nhiễu Dirichlet, chọn nước mạnh nhất) — chỉ
được publish làm best mới nếu đạt score ≥ `SP_ARENA_WIN_THRESHOLD` (mặc định
0.55). Nếu thua, giữ nguyên best cũ, generation tiếp theo vẫn self-play từ
best cũ đó. Mục đích: một generation train hỏng (overfit, data xấu, ...)
không còn âm thầm làm thụt lùi model đang dùng.

## Tận dụng T4x2

Cả self-play **và** arena đều tự phát hiện số GPU (`torch.cuda.device_count()`)
và chia việc chạy song song trên 2 GPU (mỗi worker 1 `CUDA_VISIBLE_DEVICES`).
Việc train policy/value net thì chạy trên 1 GPU (model nhỏ, không cần multi-GPU).

## Workflow các Cell
1. **Cell 1** — Cài đặt & Cấu hình (bao gồm Arena + Fresh-start)
2. **Cell 2** — Bootstrap: tìm source code + model PGN + kiểm tra resume state
3. **Cell 3** — Self-Play Pipeline (vòng lặp vô hạn, dừng khi hết giờ Kaggle hoặc đủ `SP_MAX_GEN`)


In [ ]:
# ─── Cell 1: Cài đặt & Cấu hình Self-Play ────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "chess", "-q"])

from pathlib import Path

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  CHỈNH Ở ĐÂY                                                            │
# └─────────────────────────────────────────────────────────────────────────┘
SOURCE_DATASET = "/kaggle/input/datasets/huungoc/chess-ai-source"
MODEL_DATASET  = "/kaggle/input/datasets/huungoc/chess-model"

# ── Self-Play Hyperparameters (OPTIMIZED FOR T4x2) ──────────────────────────
# Khuyến nghị dưới đây tính cho: Kaggle T4x2 (2× Tesla T4, 16GB/GPU), model
# nhỏ (~128 kênh, 20 input planes, 8x8 board), biến thể Antichess (branching
# factor thấp hơn cờ thường vì luật bắt buộc ăn quân), session Kaggle giới
# hạn ~9h. Nếu đổi hardware/board size khác thì cần tune lại.

# MCTS Configuration
# Số lần MCTS simulate mỗi nước đi lúc self-play. Cao hơn = nước đi & value
# target để train chính xác hơn, nhưng chậm gần như tuyến tính theo số này.
#   100-150 : nhanh nhất — hợp lúc test/debug pipeline, chất lượng data thấp
#   200-300 : KHUYẾN NGHỊ cho T4x2 — cân bằng tốc độ/chất lượng (mặc định 250)
#   400-800 : chuẩn AlphaZero, nhưng số ván/gen sẽ giảm mạnh trên T4 (GPU yếu
#             hơn nhiều so với TPU trong paper gốc) — chỉ nên dùng nếu bạn
#             chấp nhận ít generation hơn mỗi session.
SP_SIMULATIONS      = 250

# Data Generation
# Số ván self-play mỗi generation (tự chia đều cho 2 GPU). Nhiều ván hơn →
# data đa dạng hơn, gradient mỗi gen ổn định hơn, nhưng lâu hơn.
#   50-80   : generation nhanh — hợp giai đoạn đầu khi model còn yếu, muốn
#             lặp nhiều gen nhanh để xem xu hướng
#   100-150 : KHUYẾN NGHỊ — đủ đa dạng mà vẫn được nhiều gen/session (mặc định 100)
#   200+    : chỉ nên dùng khi model đã khá mạnh, ít cần lặp nhanh nữa
SP_GAMES_PER_GEN    = 100

# Training Configuration
# Epoch train trên replay buffer mỗi generation. Data self-play luôn MỚI mỗi
# gen (khác hẳn PGN training dùng chung 1 bộ data) nên KHÔNG cần nhiều epoch —
# train lâu trên cùng 1 batch nhỏ dễ overfit (từng gặp: PGN training overfit
# ngay sau epoch 2/20).
#   2-3     : an toàn nếu SP_GAMES_PER_GEN nhỏ (<80, ít data mỗi gen)
#   3-5     : KHUYẾN NGHỊ chung (mặc định 4)
#   6+      : chỉ hợp lý khi SP_GAMES_PER_GEN lớn (200+, đủ data để "hấp thụ")
SP_EPOCHS           = 4

# Batch size lúc train. 512 chạy ổn định trên T4 16GB với model hiện tại
# (~44MB traced). Tăng nếu muốn train nhanh hơn và không bị CUDA OOM; giảm
# (256/384) ngay nếu gặp OOM.
SP_BATCH_SIZE       = 512   # ← ĐỔI: 512 thay vì 640 (an toàn hơn)

# Learning rate cho Adam. 0.001-0.003 hợp lý cho kiểu continual fine-tuning
# này (weight giữ lại và train tiếp qua từng gen, KHÔNG train lại từ đầu mỗi
# lần) — LR cao hơn nhiều (>0.005) dễ làm model "quên" tiến bộ của các gen
# trước, thấp hơn nhiều (<0.0005) thì học chậm, tốn nhiều gen hơn mới thấy
# cải thiện rõ.
SP_LR               = 0.002

# Generation Limits
# None = chạy tới khi hết giờ Kaggle (~8h30, xem max_runtime_seconds ở Cell 3).
# Đặt số cụ thể (vd 2-3) nếu chỉ muốn chạy thử nhanh, kiểm tra pipeline chạy
# đúng (build engine, self-play, train, arena, publish) trước khi chạy full
# session — khuyến nghị làm việc này trước lần chạy đầu tiên.
SP_MAX_GEN          = None

# Exploration vs Exploitation
# Số nước đầu ván self-play chọn theo temperature=1 (sample theo visit count)
# thay vì luôn chọn nước tốt nhất — tạo đa dạng khai cuộc cho data training.
# Antichess bắt buộc ăn quân khi có thể nên branching factor nhiều thời điểm
# bị thu hẹp tự nhiên → không cần temperature_moves cao như cờ thường
# (AlphaZero gốc dùng 30).
#   10-15   : ít đa dạng hơn, các ván hội tụ giống nhau nhanh hơn
#   15-25   : KHUYẾN NGHỊ cho Antichess (mặc định 20)
#   30+     : chuẩn cờ thường, hơi thừa cho Antichess
SP_TEMPERATURE_MOVES = 20

# Resignation Policy — cho phép model tự "đầu hàng" khi thấy vị trí vô vọng,
# giúp ván kết thúc nhanh hơn (tăng ván/giờ) và data đỡ noise hơn (thay vì
# đánh tiếp vô nghĩa tới hết max_moves rồi tính hoà).
# FIXED BUG #3: Must be NEGATIVE! Resign when Q < -0.95 (losing badly)
# Logic: if (root_q < resign_thresh) then resign
# Q values: +1 (win), 0 (draw), -1 (loss)
# With 0.95 (WRONG): almost all positions have Q < 0.95 → immediate resignation
# With -0.95 (CORRECT): only resign when Q < -0.95 (truly hopeless positions)
#   resign_thresh  : KHUYẾN NGHỊ -0.90 đến -0.95 (chuẩn AlphaZero) — càng gần
#                    -1 càng "thận trọng", chỉ resign khi thực sự vô vọng
#   min_resign_ply : không cho resign trước nước này, tránh model non (đầu
#                    training, đánh giá còn kém) resign nhầm ngay khai cuộc.
#                    KHUYẾN NGHỊ 20-30.
SP_RESIGN_THRESH    = -0.95  # AlphaZero standard: -0.90 to -0.95
SP_MIN_RESIGN_PLY   = 25

# ── Fresh start ──────────────────────────────────────────────────────────────
# True  = LUÔN train self-play từ gen 1 với model khởi đầu = INITIAL_MODEL
#         (model PGN tìm thấy ở Cell 2), bỏ qua mọi resume_state.json cũ còn
#         sót lại trong OUTPUT_DIR (vd nếu bạn Rerun Cell 1→3 nhiều lần trong
#         cùng 1 session Kaggle). Output cũ (nếu có) sẽ được dời qua thư mục
#         _previous_run_<timestamp>/ chứ không bị xoá.
# False = hành vi cũ: tự resume nếu thấy resume_state.json trong OUTPUT_DIR.
SP_FORCE_FRESH_START = True

# ── Arena Gating ─────────────────────────────────────────────────────────────
# Model mới train xong phải THẮNG model "best" hiện tại trong 1 trận đấu thì
# mới được publish thành best_model_traced.pt (dùng cho self-play + xuất ra).
# Trước đây pipeline ghi đè best_model_traced.pt vô điều kiện sau mỗi gen, kể
# cả khi gen đó thực ra yếu hơn (overfit / data xấu) — âm thầm làm thụt lùi
# model. Giờ có bước đấu N ván (không nhiễu Dirichlet, chọn nước tốt nhất,
# tự chia đều cho 2 GPU) trước khi quyết định promote hay giữ nguyên best cũ.
#
# Khuyến nghị (T4x2):
#   SP_ARENA_GAMES: 30-40 đủ cho tín hiệu thắng/thua tương đối tin cậy mà
#     không chiếm quá nhiều thời gian mỗi gen (mặc định 40). Nếu thấy quyết
#     định promote/reject có vẻ "nhiễu" (model rõ ràng tốt hơn vẫn hay bị
#     reject, hoặc ngược lại) → tăng lên 60-100 để giảm phương sai đo lường.
#   SP_ARENA_SIMULATIONS: thường để THẤP HƠN SP_SIMULATIONS một chút — arena
#     chỉ cần đủ mạnh để phân biệt model nào hơn, không cần chính xác tuyệt
#     đối như lúc sinh training data (mặc định 200 so với SP_SIMULATIONS=250).
#   SP_ARENA_WIN_THRESHOLD = 0.55 là con số gốc AlphaGo Zero dùng để quyết
#     định promote — giữ nguyên trừ khi có lý do cụ thể để đổi. Cao hơn
#     (0.60+) = khắt khe hơn, model tiến chậm nhưng chắc; thấp hơn (0.50-0.52)
#     gần như quay lại hành vi "luôn publish" cũ.
SP_ARENA_ENABLED        = True
SP_ARENA_GAMES          = 40     # số ván đấu mỗi gen
SP_ARENA_SIMULATIONS    = 200    # sims/nước lúc đấu (thấp hơn lúc train để đấu nhanh)
SP_ARENA_WIN_THRESHOLD  = 0.55   # candidate cần score (thắng + 0.5*hòa)/tổng ván >= giá trị này
SP_ARENA_MAX_MOVES      = 200

# ── Paths ────────────────────────────────────────────────────────────────────
WORKDIR    = Path("/kaggle/working/chess_selfplay")
OUTPUT_DIR = Path("/kaggle/working/chess_outputs")
WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "checkpoints").mkdir(exist_ok=True)

# ── Summary ──────────────────────────────────────────────────────────────────
print("✓ Cell 1 done — Cấu hình OPTIMIZED cho T4x2:")
print(f"  simulations={SP_SIMULATIONS}  games/gen={SP_GAMES_PER_GEN}  epochs/gen={SP_EPOCHS}")
print(f"  batch_size={SP_BATCH_SIZE}  lr={SP_LR}")
print(f"  max_gen={SP_MAX_GEN}  temperature_moves={SP_TEMPERATURE_MOVES}")
print(f"  resign_thresh={SP_RESIGN_THRESH}  min_resign_ply={SP_MIN_RESIGN_PLY}")
print(f"  arena: enabled={SP_ARENA_ENABLED}  games={SP_ARENA_GAMES}  sims={SP_ARENA_SIMULATIONS}  win_threshold={SP_ARENA_WIN_THRESHOLD}")
print(f"  OUTPUT: {OUTPUT_DIR}")
print(f"\n⏱️  Ước tính runtime: ~9h cho {SP_MAX_GEN} generations")
if SP_MAX_GEN is not None:
    print(f"📊 Total data: ~{SP_GAMES_PER_GEN * SP_MAX_GEN} games = ~{SP_GAMES_PER_GEN * SP_MAX_GEN * 40} positions")
else:
    print(f"📊 Infinite mode: ~{SP_GAMES_PER_GEN} games per generation")


In [ ]:
# ─── Cell 2: Bootstrap — tìm source code & model ──────────────────────────────
import glob, shutil
from pathlib import Path

# ── Tìm PROJECT_ROOT ─────────────────────────────────────────────────────────
def find_project_root(dataset_path: str) -> Path:
    for pattern in [
        f"{dataset_path}/Chess-AI-out",
        f"{dataset_path}/*/Chess-AI-out",
        f"{dataset_path}",
    ]:
        for m in glob.glob(pattern):
            p = Path(m)
            if (p / "AI/engine/selfplay.cpp").exists() or \
               (p / "AI/src/colab_selfplay_pipeline.py").exists():
                return p
    raise FileNotFoundError(f"Không tìm thấy project root trong {dataset_path}")

try:
    PROJECT_ROOT = find_project_root(SOURCE_DATASET)
    print(f"[Bootstrap] ✓ PROJECT_ROOT = {PROJECT_ROOT}")
except FileNotFoundError as e:
    print(f"[ERROR] {e}")
    print("[HELP] Hãy thêm dataset chứa source code Chess AI vào kaggle")
    raise

# ── Add pipeline vào sys.path ─────────────────────────────────────────────────
import sys
pipeline_dir = PROJECT_ROOT / "AI" / "src"
if str(pipeline_dir) not in sys.path:
    sys.path.insert(0, str(pipeline_dir))
print(f"[Bootstrap] Pipeline dir: {pipeline_dir}")

# ── Tìm model khởi đầu ───────────────────────────────────────────────────────
INITIAL_MODEL = None
model_candidates = []

# 1. Từ model dataset
if Path(MODEL_DATASET).exists():
    model_candidates += [
        f"{MODEL_DATASET}/best_model_traced.pt",
        f"{MODEL_DATASET}/chess_pgn_train/best_model_traced.pt",
    ]
    model_candidates += sorted(glob.glob(f"{MODEL_DATASET}/**/*.pt", recursive=True))

# 2. Từ source dataset (nếu đã save model vào đó)
model_candidates += [
    f"{SOURCE_DATASET}/Chess-AI-out/AI/data/best_model_traced.pt",
    f"{SOURCE_DATASET}/best_model_traced.pt",
]

# 3. Từ output lần trước
model_candidates += sorted(
    glob.glob(str(OUTPUT_DIR / "model_gen_*.pt")),
    key=lambda p: int(Path(p).stem.split("_")[-1]) if Path(p).stem.split("_")[-1].isdigit() else 0,
    reverse=True
)

for c in model_candidates:
    if Path(c).exists():
        INITIAL_MODEL = Path(c)
        size_mb = INITIAL_MODEL.stat().st_size / 1e6
        print(f"[Bootstrap] ✓ Model tìm thấy: {INITIAL_MODEL}  ({size_mb:.1f}MB)")
        break

if INITIAL_MODEL is None:
    print("[Bootstrap] ⚠️  Không tìm thấy model → self-play sẽ dùng random init")
    print("[Bootstrap] Lưu ý: Model random sẽ chơi rất tệ ở đầu, cần nhiều gen để cải thiện")

# Kiểm tra resume state
resume_state_path = OUTPUT_DIR / "resume_state.json"

if SP_FORCE_FRESH_START and resume_state_path.exists():
    # Có state cũ nhưng người dùng muốn train lại từ đầu (vd: dùng model PGN
    # mới) → dời toàn bộ OUTPUT_DIR cũ sang thư mục lưu trữ, không resume.
    import time as _time
    archive_dir = OUTPUT_DIR.parent / f"_previous_run_{_time.strftime('%Y%m%d_%H%M%S')}"
    print(f"[Bootstrap] ⚠️  SP_FORCE_FRESH_START=True — có resume_state.json cũ nhưng sẽ BỎ QUA.")
    print(f"[Bootstrap]   Dời output cũ sang: {archive_dir}")
    shutil.move(str(OUTPUT_DIR), str(archive_dir))
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "checkpoints").mkdir(exist_ok=True)
    RESUME = False
    print("\n[Bootstrap] Chạy mới hoàn toàn (fresh start, dùng INITIAL_MODEL ở trên)")
elif resume_state_path.exists():
    import json
    with open(resume_state_path) as f:
        rs = json.load(f)
    print(f"\n[Bootstrap] ✓ Resume state tìm thấy: gen={rs.get('generation', 0)}")
    print(f"[Bootstrap]   Best model: {rs.get('best_model_path', 'N/A')}")
    RESUME = True
else:
    print("\n[Bootstrap] Chạy mới (không có resume state)")
    RESUME = False

print("\n✓ Cell 2 done")


In [ ]:
# ─── Cell 3: Self-Play Pipeline ───────────────────────────────────────────────
# Pipeline vòng lặp:
#   1. Build C++ engine (tự động, cache nếu source không đổi)
#   2. Self-play N games → replay buffer .bin
#   3. Train policy+value net trên replay buffer
#   4. Export TorchScript model mới
#   5. Lặp lại từ bước 2 với model mới
#
# Dừng tự nhiên khi Kaggle timeout (~9h) hoặc đủ SP_MAX_GEN generations.
# Lần sau: thêm OUTPUT_DIR vào dataset → Resume tự động từ gen đã làm.
#
# Output files:
#   OUTPUT_DIR/selfplay_gen_N.bin        — replay data gen N
#   OUTPUT_DIR/model_gen_N.pt            — model vừa train ở gen N (trước khi qua arena)
#   OUTPUT_DIR/arena_gen_N.json          — kết quả đấu model_gen_N vs best hiện tại
#   WORKDIR/best_model_traced.pt         — model tốt nhất ĐÃ ĐƯỢC PROMOTE (dùng cho self-play)
#   OUTPUT_DIR/checkpoints/latest_checkpoint.pt — checkpoint (model+optimizer) để resume, ghi đè mỗi epoch
#   OUTPUT_DIR/resume_state.json         — resume state
#
# Lưu ý: best_model_traced.pt nằm ở WORKDIR (không phải OUTPUT_DIR) vì
# best_model_path_override = WORKDIR / "best_model_traced.pt" ở dưới.

import importlib.util
from pathlib import Path

spec = importlib.util.spec_from_file_location(
    "colab_selfplay_pipeline",
    PROJECT_ROOT / "AI" / "src" / "colab_selfplay_pipeline.py"
)
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)

print(f"\n{'='*60}")
print(f"SELF-PLAY PIPELINE — {'Resume' if RESUME else 'Fresh start'}")
print(f"  simulations   = {SP_SIMULATIONS}")
print(f"  games/gen     = {SP_GAMES_PER_GEN}")
print(f"  epochs/gen    = {SP_EPOCHS}")
print(f"  max_gen       = {'∞' if SP_MAX_GEN is None else SP_MAX_GEN}")
print(f"  temp_moves    = {SP_TEMPERATURE_MOVES}")
print(f"  output        = {OUTPUT_DIR}")
print(f"{'='*60}\n")

# ── Time limit: 8h30m (30600s) khi SP_MAX_GEN = None ───────────────────────
max_runtime_seconds = 30600 if SP_MAX_GEN is None else None  # 8h30m = 8.5 * 3600
if max_runtime_seconds:
    print(f"⏱️  Time limit: {max_runtime_seconds / 3600:.1f}h khi chạy infinite mode")
# ────────────────────────────────────────────────────────────────────────────

pipeline.run_pipeline(
    project_root             = PROJECT_ROOT,
    workdir                  = WORKDIR,
    drive_root               = OUTPUT_DIR,
    best_model_path_override = WORKDIR / "best_model_traced.pt",
    initial_model_path       = INITIAL_MODEL,
    simulations              = SP_SIMULATIONS,
    games_per_generation     = SP_GAMES_PER_GEN,
    epochs                   = SP_EPOCHS,
    batch_size               = SP_BATCH_SIZE,
    learning_rate            = SP_LR,
    max_generations          = SP_MAX_GEN,
    infinite                 = (SP_MAX_GEN is None),
    resume                   = RESUME,
    temperature_moves        = SP_TEMPERATURE_MOVES,
    log_every_games          = max(5, SP_GAMES_PER_GEN // 20),
    heartbeat_seconds        = 60,
    resign_thresh            = SP_RESIGN_THRESH,
    min_resign_ply           = SP_MIN_RESIGN_PLY,
    max_runtime_seconds      = max_runtime_seconds,
    arena_enabled            = SP_ARENA_ENABLED,
    arena_games               = SP_ARENA_GAMES,
    arena_simulations         = SP_ARENA_SIMULATIONS,
    arena_win_threshold       = SP_ARENA_WIN_THRESHOLD,
    arena_max_moves           = SP_ARENA_MAX_MOVES,
)
